# 03 — Silver para Gold

**Arquivo:** 03-silver-para-gold.ipynb
**Objetivo:** agregar a Silver em tabelas analíticas de vacância e evolução do segmento
**Entrada:** dados/silver/imoveis/ (Parquet particionado por ano)
**Saída:** dados/gold/*.parquet
**Onde roda:** local (.venv, DuckDB) — **Camada:** Silver → Gold

In [1]:
# duckdb executa SQL sobre arquivos Parquet sem precisar de servidor de banco.
import duckdb
import pandas as pd
from pathlib import Path

# Caminhos centralizados. Na migração para a nuvem, só estas linhas mudam.
RAIZ = Path("..").resolve()
SILVER = RAIZ / "dados" / "silver"
GOLD = RAIZ / "dados" / "gold"

# Abre um banco DuckDB em memória: existe só enquanto o notebook estiver aberto.
con = duckdb.connect()

# Caminho de todos os Parquet da Silver, em qualquer subpasta de ano.
# .as_posix() troca as barras do Windows (\) por barras normais (/),
# que o DuckDB e os serviços de nuvem entendem sem ambiguidade.
caminho_silver = (SILVER / "imoveis").as_posix() + "/**/*.parquet"

# Cria a view. hive_partitioning = true reconstrói a coluna "ano"
# a partir do nome das pastas ano=2020, ano=2021 etc.
con.execute(f"""
    CREATE OR REPLACE VIEW silver_imoveis AS
    SELECT * FROM read_parquet('{caminho_silver}', hive_partitioning = true)
""")

# Conferência: a view precisa enxergar os mesmos 5.607 registros gravados.
total = con.sql("SELECT COUNT(*) FROM silver_imoveis").fetchone()[0]
print("Registros visíveis na view:", total)
print("Bate com a Silver (5607)?", total == 5607)

Registros visíveis na view: 5607
Bate com a Silver (5607)? True


In [2]:
# Agrupa todos os imóveis de cada trimestre e calcula os indicadores do segmento.
# vacancia_ponderada é o "Jeito 2": soma das áreas vagas dividida pela soma das áreas.
# vacancia_media_simples é o "Jeito 1", incluído apenas para comparação.
consulta = """
    SELECT
        Data_Referencia,
        ano,
        Trimestre,
        COUNT(DISTINCT CNPJ_Fundo_Classe)   AS qtd_fundos,
        COUNT(*)                            AS qtd_registros,
        SUM(Area)                           AS area_total_m2,
        SUM(Area_Vaga_m2)                   AS area_vaga_m2,
        SUM(Area_Vaga_m2) / SUM(Area)       AS vacancia_ponderada,
        AVG(Percentual_Vacancia)            AS vacancia_media_simples
    FROM silver_imoveis
    GROUP BY Data_Referencia, ano, Trimestre
    ORDER BY Data_Referencia
"""

# Executa a consulta e converte o resultado em tabela do pandas.
df_gold_segmento = con.sql(consulta).df()

# Grava a Gold. Arquivo único: to_parquet sobrescreve o anterior, então rodar
# de novo não duplica nada.
GOLD.mkdir(parents=True, exist_ok=True)
df_gold_segmento.to_parquet(GOLD / "vacancia_segmento_trimestral.parquet", index=False)

print("Trimestres na série:", len(df_gold_segmento))
df_gold_segmento

Trimestres na série: 35


,Data_Referencia,ano,Trimestre,qtd_fundos,qtd_registros,area_total_m2,area_vaga_m2,vacancia_ponderada,vacancia_media_simples
0,2020-03-31,2020,1,23,99,4.787511e+06,233123.349824,0.048694,0.068007
1,2020-06-30,2020,2,25,108,4.681332e+06,196819.186400,0.042043,0.063508
2,2020-09-30,2020,3,25,114,5.078611e+06,150754.873557,0.029684,0.051050
3,2020-12-31,2020,4,24,124,5.561860e+06,274071.186489,0.049277,0.073195
4,2021-03-31,2021,1,26,137,7.060462e+06,296456.054023,0.041988,0.064903
5,2021-06-30,2021,2,29,145,6.610305e+06,349096.050475,0.052811,0.067418
6,2021-09-30,2021,3,29,149,7.213286e+06,365445.889921,0.050663,0.057300
7,2021-12-31,2021,4,32,158,7.788187e+06,370654.374789,0.047592,0.053201
8,2022-03-31,2022,1,31,225,7.910103e+06,275957.708632,0.034887,0.033396
9,2022-06-30,2022,2,34,240,8.254590e+06,180165.018050,0.021826,0.031074
